In [ ]:
# Ensure gradalg is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# gradalg isn't already installed into this kernel.
try:
    import gradalg  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "gradalg" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import gradalg  # noqa: F401

# 08 — Birleşik Tablo

Bu notebook [08_unified_picture.md](08_unified_picture.md) markdown'ının çalıştırılabilir sürümüdür. Tek hipotez `[π, π]_SN = 0` — fonksiyon ve form seviyelerinde aynı obstruction. Theorem Book citation zinciri. Paralel örnek: `dH = 0`.

## Tek hipotez, iki yüz

`jacobi_condition` ve `koszul_jacobi_condition` aynı `Expr`'e işaret ediyor — sadece display ismi farklı.

In [ ]:
from gradalg.library.declarations import Bivector, Forms, Functions
from gradalg.library.poisson import PoissonBracket
from gradalg.core.registry import PropertyRegistry

reg = PropertyRegistry()
pi = Bivector('π', registry=reg)
poisson = PoissonBracket.from_bivector(pi)

c1 = poisson.jacobi_condition(reg)
c2 = poisson.koszul_jacobi_condition(reg)
print('func obs:', c1.obstruction)
print('form obs:', c2.obstruction)
print('same expr?:', c1.obstruction == c2.obstruction)

## Fonksiyon vs form — aynı obstruction'da kesişim

In [ ]:
f, g, h = Functions('f g h', degree=-1, registry=reg)
alpha, beta, gamma = Forms('α β γ', degree=1, registry=reg)

func_chain = poisson.prove_jacobi_reduction(f, g, h, registry=reg)
form_chain = poisson.prove_koszul_jacobi_reduction(
    alpha, beta, gamma, registry=reg
)

print('func rule:', func_chain.steps[0].rule)
print('form rule:', form_chain.steps[0].rule)
print('func after:', func_chain.steps[0].after)
print('form after:', form_chain.steps[0].after)
print('same after?:', func_chain.steps[0].after == form_chain.steps[0].after)

## Klasik-derived köprü — `reflexive` step

Paketin expand kuralları her iki tarafı aynı `Expr` ağacına getirir; Koszul eşdeğerliği yapısal olarak kapanır.

In [ ]:
chain_eq = poisson.prove_koszul_equivalence(alpha, beta, registry=reg)
print('len:', len(chain_eq), 'rule:', chain_eq.steps[0].rule)

## Seeded teoremler — citation zinciri

In [ ]:
from gradalg.library import theorem_book

for name in (
    'poisson_jacobi',
    'poisson_koszul_equivalence',
    'poisson_koszul_jacobi',
):
    thm = theorem_book.get(name)
    print(name)
    for ax in thm.from_axioms:
        print('   -', ax)

## `dH = 0` — Courant tarafı aynı desen

In [ ]:
from gradalg.brackets.courant import CourantBracket
from gradalg.core.expr import Symbol
from gradalg.core.properties import Graded

reg_h = PropertyRegistry()
H = Symbol('H')
reg_h.declare(H, Graded(degree=3))

C = CourantBracket(background_H=H)
cond = C.jacobi_condition(reg_h)
print('name       :', cond.name)
print('obstruction:', cond.obstruction)

twist = theorem_book.get('courant_jacobi_twist')
print('twist from_axioms:', twist.from_axioms)

## Sonraki adım

Aksiyomun *kendisi* nereden geliyor? [09_foundations.md](09_foundations.md).